In [0]:
print("Hello World")

In [0]:
%sql
DROP SCHEMA IF EXISTS 03_gold.gold CASCADE;

In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS 03_gold.gold;

In [0]:
%sql
CREATE TABLE 03_gold.gold.dim_customer AS
SELECT
    customer_id,
    customer_name,
    customer_type,
    country,
    signup_date
FROM 02_silver.silver.customer;

In [0]:
%sql
CREATE TABLE 03_gold.gold.dim_products AS
SELECT
    product_id,
    product_name,
    category,
    cost_price
FROM 02_silver.silver.products;

In [0]:
%sql
CREATE TABLE 03_gold.gold.dim_regions AS
SELECT
    country,
    region_name,
    region_code
FROM 02_silver.silver.regions;

In [0]:
%sql    
CREATE OR REPLACE TABLE 03_gold.gold.fact_invoice_line_item AS
SELECT
    il.invoice_line_id                                 AS invoice_line_id,
    il.invoice_id                                      AS invoice_id,
    il.product_id                                      AS product_id,
    il.quantity                                        AS quantity,
    CAST(il.unit_price AS DECIMAL(18,2))               AS unit_price,
    CAST(il.discount AS DECIMAL(18,2))                 AS discount,
    i.customer_id                                      AS customer_id,
    i.invoice_date                                     AS invoice_date,
    i.invoice_month                                    AS invoice_month,
    i.currency                                         AS currency,
    i.region                                           AS region,
    i.invoice_status                                   AS invoice_status,
    CAST(i.rate_to_usd AS DECIMAL(18,6))               AS rate_to_usd,
    ROUND(
        (CAST(il.unit_price AS DECIMAL(18,2)) - CAST(il.discount AS DECIMAL(18,2)))
        * CAST(il.quantity AS INT),
    2) AS line_revenue_local,
    ROUND(
        (CAST(il.unit_price AS DECIMAL(18,2)) - CAST(il.discount AS DECIMAL(18,2)))
        * CAST(il.quantity AS INT)
        * CAST(i.rate_to_usd AS DECIMAL(18,6)),
    2) AS line_revenue_usd
FROM 02_silver.silver.invoice_line_items il
LEFT JOIN 02_silver.silver.invoices i ON il.invoice_id = i.invoice_id;

In [0]:
%sql
drop table 03_gold.gold.fact_payment

In [0]:
%sql
CREATE OR REPLACE TABLE 03_gold.gold.fact_payment AS
SELECT
    p.payment_id,
    p.invoice_id,
    p.payment_date,
    CAST(p.payment_amount AS DECIMAL(18,2)) AS payment_amount,
    p.payment_method,
    i.customer_id,
    i.invoice_status
FROM 02_silver.silver.payments p
LEFT JOIN 02_silver.silver.invoices i ON p.invoice_id = i.invoice_id;

In [0]:
%sql
select * from 03_gold.gold.dim_customer

In [0]:
%sql
select * from 03_gold.gold.dim_products

In [0]:
%sql
select * from 03_gold.gold.dim_regions

In [0]:
%sql
select * from 03_gold.gold.fact_invoice_line_item

In [0]:
%sql
select * from 03_gold.gold.fact_payment